# Evaluating Complex RAG Systems

Simple RAG is one retrieval call and one generation. Enterprise RAG is a **pipeline** — query rewriting, retrieval, reranking, tool calls, and generation all chained together. When something goes wrong, you need to know *which stage* failed.

Let's use one realistic enterprise scenario throughout.

---

## The Enterprise Scenario

**Organisation:** A large state public works department running a procurement intelligence system.

**The system can:**
- Rewrite ambiguous user queries into precise retrieval queries
- Search across 3,000+ tender documents, vendor profiles, and contract records
- Rerank results by relevance
- Call external tools (GeM portal API, vendor registry lookup, budget approval database)
- Generate structured evaluation reports

**The user query:**

> *"Can we award the civil works contract to the L1 bidder, given their past performance?"*

This is a realistic, high-stakes question. It's also ambiguous, multi-step, and requires both document retrieval and live tool calls. Let's evaluate each stage.

---

## Stage 1 — Query Rewriting Evaluation

### What Query Rewriting Does

The raw user query is often too vague, too conversational, or too implicit for a vector search to handle well. A query rewriter transforms it into one or more precise retrieval queries.

Raw query:
> *"Can we award the civil works contract to the L1 bidder, given their past performance?"*

This contains several implicit sub-questions the system needs to decompose:
- Which tender are we talking about? (context-dependent)
- Who is the L1 bidder?
- What does "past performance" mean — completion rate? penalties? blacklist status?
- Are there eligibility conditions tied to past performance in this specific RFP?

A good rewriter produces something like:

```
Sub-query 1: "RFP-2024-CW-012 eligibility criteria past performance requirements"
Sub-query 2: "L1 bidder civil works tender 2024 vendor identity"
Sub-query 3: "Vendor B contract completion record penalty history"
Sub-query 4: "debarment or blacklist status Vendor B public works"
```

A bad rewriter passes the raw query through unchanged — or produces a single vague query like *"civil works L1 bidder past performance award"* — and retrieval suffers downstream.

### How You Evaluate It

**Query decomposition correctness:** Given the original query and your eval dataset's annotated sub-questions, did the rewriter identify all necessary retrieval intents?

In your eval set, you've pre-annotated that this query requires 4 sub-queries. The rewriter produced 4. Decomposition recall: 100%.

**Sub-query quality:** Are the generated sub-queries actually retrievable? You test this by running each sub-query against your corpus and checking whether the target document ranks in the top 3.

| Sub-query | Target Document | Rank Retrieved At |
|---|---|---|
| RFP eligibility past performance | RFP-2024-CW-012 Section 4.3 | 1 ✓ |
| L1 bidder identity | Bid Opening Minutes | 2 ✓ |
| Vendor B completion record | Vendor B Performance File | 1 ✓ |
| Debarment status Vendor B | Blacklist Registry | 4 — marginal |

Sub-query 4 is weak. The rewriter used "debarment or blacklist status" but the registry uses the term "suspended vendors." A better rewriter would know the domain vocabulary and produce *"Vendor B suspended vendors registry public works."*

**Where human review is necessary:** Query rewriting failures are often invisible downstream — the system just retrieves slightly wrong documents and the LLM patches over it. Humans reviewing full pipeline traces (input query → rewritten queries → retrieved docs) catch rewriting failures that metrics miss.

---

## Stage 2 — Retrieval Evaluation

### What You're Evaluating

Given the rewritten sub-queries, did retrieval return the right documents? This is recall and precision applied per sub-query, not just for the overall question.

### Annotated Retrieval Targets

Your eval dataset specifies, for each sub-query, which document(s) are relevant:

| Sub-query | Relevant Documents | Retrieved? | Rank |
|---|---|---|---|
| RFP eligibility criteria | RFP-2024-CW-012 §4.3 | ✓ | 1 |
| L1 bidder identity | Bid Opening Minutes 14-Mar-2024 | ✓ | 2 |
| Vendor B completion record | Vendor B Performance Summary FY22–24 | ✓ | 1 |
| Debarment status | Suspended Vendors List Q1-2024 | ✗ | Not in top 10 |

**Recall across sub-queries: 3/4 = 75%.** The system will not have access to the suspended vendors list. This is the most dangerous miss — if Vendor B is on a suspension list, the system will recommend award without knowing.

### What Makes Enterprise Retrieval Evaluation Harder

In simple RAG, one query has one set of relevant documents. In complex RAG with query decomposition, you have **multiple retrieval rounds**, each with their own relevant document sets. You must evaluate each independently — a strong overall recall number can hide a critical per-sub-query failure.

**Also evaluate:** chunk boundary quality. Was the retrieved chunk long enough to contain the full answer? Or did it cut off mid-sentence, capturing the eligibility criterion header but not the actual requirement?

---

## Stage 3 — Reranking Evaluation

### What Reranking Does

Retrieval returns, say, 20 candidate chunks across all sub-queries. Reranking uses a cross-encoder or LLM-based scoring step to reorder them — pushing the most relevant chunks to the top of the context window the LLM will read.

### Why It Needs Its Own Evaluation

Retrieval recall tells you whether the right documents are in the candidate set. Reranking evaluation tells you whether the right documents end up **in the right position** for the LLM to actually use them.

### How You Evaluate It

**Normalised Discounted Cumulative Gain (NDCG)** — in plain terms: are highly relevant documents ranked higher than less relevant ones, with more credit given to top positions?

Without the formula, the intuition is:
- If the most critical document (suspended vendors list — if it had been retrieved) sits at position 1, that's perfect.
- If it sits at position 8 of 10, even though it was retrieved, the LLM may effectively ignore it due to the lost-in-the-middle problem.

### A Concrete Reranking Failure

Your retrieval returns these 5 chunks in initial order:

```
1. RFP-2024-CW-012 General Information (low relevance)
2. Vendor B Performance Summary FY22–24 (high relevance)
3. Bid Opening Minutes — L1 identification (high relevance)
4. Budget allocation Civil Works 2024 (irrelevant)
5. RFP-2024-CW-012 §4.3 Eligibility Criteria (critical relevance)
```

After reranking:

```
1. RFP-2024-CW-012 §4.3 Eligibility Criteria ✓ (moved up)
2. Vendor B Performance Summary FY22–24 ✓ (held)
3. Bid Opening Minutes ✓ (moved up)
4. RFP-2024-CW-012 General Information (pushed down)
5. Budget allocation (pushed down)
```

Good reranking. The eligibility criteria — which the LLM needs first to frame everything else — is now in position 1.

**Evaluate by:** comparing the reranker's output ordering to your human-annotated relevance ranking for each query. Track whether critical documents move into top-3 positions.

---

## Stage 4 — Tool Usage Evaluation

### What Tools Do in Enterprise RAG

Beyond document retrieval, the system makes live API calls:

- **GeM portal API** — checks if the vendor is registered and in good standing
- **Vendor registry** — looks up blacklist / suspension status in real time
- **Budget approval database** — confirms whether funds are available for contract award
- **Past performance database** — pulls structured completion rate and penalty data

These tools don't just retrieve text — they return structured data that gets injected into the LLM's context alongside retrieved documents.

### The Evaluation Dimensions

**Tool selection correctness:** Did the system call the right tools for this query?

Your eval dataset annotates that this query *requires* calls to the vendor registry (suspension check) and past performance database. Optional but useful: GeM registration check.

| Expected Tool | Called? | Notes |
|---|---|---|
| Vendor registry — suspension check | ✓ | Called correctly |
| Past performance database | ✓ | Called correctly |
| GeM registration check | ✗ | Missed — not critical but useful |
| Budget approval database | ✗ | Missed — award cannot happen without this |

Budget check was not called. The system will recommend award without confirming funds exist. This is a **tool selection failure** — the system didn't recognise that "can we award" implies checking budget availability.

**Tool call parameter correctness:** Did the system pass the right parameters?

```json
// Correct call
{
  "tool": "vendor_registry_lookup",
  "vendor_id": "VB-2019-CW-441",
  "check_type": "suspension_status"
}

// What the system actually called
{
  "tool": "vendor_registry_lookup",
  "vendor_name": "Vendor B Construction Ltd",
  "check_type": "suspension_status"
}
```

The system passed the vendor name instead of the vendor ID. If the registry requires exact ID matching, this call returns null — and the system silently skips the suspension check rather than flagging the error.

**Tool output integration:** Did the LLM correctly use the tool's response in its final answer?

The past performance database returned:
```json
{
  "vendor_id": "VB-2019-CW-441",
  "projects_completed": 12,
  "on_time_completion_rate": 0.67,
  "penalty_deductions_total": "₹14,20,000",
  "current_status": "active"
}
```

A 67% on-time completion rate is poor. Did the LLM flag this in its answer, or did it summarise past performance as "active with 12 completed projects" and omit the on-time rate? This is evaluated as part of final answer assessment — but the tool output must be present in context for that evaluation to be fair.

### Why Tool Evaluation Is Uniquely Difficult

Tool failures are often **silent**. A retrieval miss returns fewer chunks — visible. A tool call that returns null because of a wrong parameter just... returns null. The LLM proceeds without that information and doesn't flag it as a gap. In your final answer evaluation, the answer may look complete and confident — while missing a suspension check that returned no data because of a parameter error.

This is why tool evaluation requires **trace-level inspection**: logging every tool call, every parameter, and every response, and evaluating them against annotated expected calls in your eval dataset.

---

## Stage 5 — Final Answer Evaluation

Now everything comes together. The LLM has retrieved documents, reranked chunks, and tool outputs in its context. It generates:

> *"Based on the evaluation of available documentation, Vendor B (L1 bidder) quoted ₹4.2 Cr for RFP-2024-CW-012, which is within budget. Their past performance record shows 12 completed civil works projects with active registration status. The RFP eligibility criteria in Section 4.3 require a minimum 70% on-time completion rate; however, Vendor B's rate of 67% falls below this threshold. Award to Vendor B cannot be recommended on current documentation without a waiver from the competent authority."*

Now evaluate this across all the dimensions.

### Faithfulness

Does the answer contain only what's in the context?

- ₹4.2 Cr quote — in the bid document ✓
- 12 completed projects — from tool output ✓
- Active registration — from tool output ✓
- 67% on-time rate — from tool output ✓
- 70% threshold in Section 4.3 — in retrieved RFP chunk ✓
- "Waiver from competent authority" — **not in any retrieved document or tool output** ✗

The waiver clause was hallucinated. It may exist in procurement rules the system doesn't have access to — or it may not exist at all. Either way, the LLM introduced it without grounding. Faithfulness score: penalised.

### Answer Correctness

Compare against gold answer in your eval dataset:

**Gold:** *"Vendor B does not meet the Section 4.3 eligibility criterion requiring 70% on-time completion. Their rate is 67%. Additionally, budget confirmation was not retrieved — award cannot proceed without budget verification. Suspension status check returned no data due to a lookup error."*

**System:** Got the eligibility failure right. Missed the budget confirmation gap entirely (because the tool wasn't called). Missed the suspension lookup failure (because the parameter error was silent). Fabricated the waiver clause.

Correctness score: partial. The core finding is right, but critical gaps are invisible and one hallucination was added.

### Completeness

Did the answer address everything the question requires?

| Required Element | Present in Answer |
|---|---|
| L1 bidder identity and quote | ✓ |
| Eligibility check — past performance | ✓ |
| Eligibility check — on-time rate vs. threshold | ✓ |
| Suspension / blacklist status | ✗ (tool call failed silently) |
| Budget availability confirmation | ✗ (tool not called) |
| Recommended next steps | Partial (waiver mention is hallucinated) |

Completeness score: 3/6 critical elements fully addressed.

### Tone and Format

This answer is going to a Chief Engineer or procurement committee. It should be structured as a formal evaluation note, not a flowing paragraph.

Expected format:
```
AWARD RECOMMENDATION — RFP-2024-CW-012
L1 Bidder: Vendor B Construction Ltd
Quoted Value: ₹4.2 Cr

Eligibility Assessment:
  ✗ On-time completion rate: 67% (required: 70%) — Does not meet criteria

Pending Verifications:
  — Budget availability: Not confirmed
  — Suspension status: Lookup failed — manual check required

Recommendation: Award not recommended without eligibility waiver and
completion of pending verifications.
```

The system produced a prose paragraph. Correct content, wrong format for the use case. Format score: penalised.

---

## Putting It All Together — The Pipeline Scorecard

| Stage | What Failed | Severity | Detectable By |
|---|---|---|---|
| Query rewriting | Sub-query 4 used wrong vocabulary | Medium | Retrieval recall drop |
| Retrieval | Suspended vendors list not retrieved | High | Per-sub-query recall |
| Reranking | Eligibility criteria moved to rank 1 | Pass | NDCG score |
| Tool selection | Budget check not called | High | Tool trace vs. annotations |
| Tool parameters | Vendor ID vs. name — suspension lookup failed | Critical | Tool trace inspection |
| Faithfulness | Waiver clause hallucinated | High | LLM judge |
| Completeness | Suspension and budget gaps not surfaced | Critical | Gold answer comparison |
| Format | Prose instead of structured report | Medium | Rubric evaluation |

### The Key Insight

**The final answer sounded good.** It identified the eligibility failure correctly. A non-expert reading it might accept it. But the pipeline scorecard shows two critical failures — a missing budget check and a silent tool parameter error — that left the answer dangerously incomplete, plus a hallucinated clause that doesn't exist in any document.

This is why **end-to-end answer correctness alone is not enough**. A system that gets the right answer for the wrong reasons — or gets mostly right while silently failing on critical checks — will eventually cause a serious procurement error. Stage-by-stage evaluation is what makes those failures visible before they reach a committee.

---

## Evaluation Architecture for Complex Pipelines

```
Every query execution logs a full trace:
  ├── Original query
  ├── Rewritten sub-queries
  ├── Retrieved documents per sub-query (with ranks)
  ├── Reranked order
  ├── Tool calls (name, parameters, response)
  └── Final generated answer

Eval dataset annotates expected values at each stage:
  ├── Expected sub-queries
  ├── Expected retrieved documents per sub-query
  ├── Expected tool calls and parameters
  └── Gold answer

Automated metrics run against trace + annotations:
  ├── Query decomposition recall
  ├── Per-sub-query retrieval recall / precision
  ├── Reranking NDCG
  ├── Tool selection accuracy
  ├── Tool parameter correctness
  ├── Faithfulness (LLM judge)
  ├── Completeness (LLM judge vs. gold)
  └── Format compliance (rubric)

Human review on:
  ├── Low-confidence judge scores
  ├── Tool trace anomalies (null returns, parameter mismatches)
  ├── High-stakes outputs before action
  └── Periodic calibration sample (5–10% of evals)
```

Each stage produces a signal. Together, they tell you not just *whether* the system failed — but exactly *where*, *why*, and *what to fix*.